# DataGuard: NYC 311 Data Issues Report

## Notebook Purpose

This notebook is the fourth notebook in the DataGuard project.

The goal of Day 4 is to detect data quality issues beyond missing values in the NYC 311 dataset.

This notebook focuses on:

- Detecting duplicate rows and duplicate request IDs
- Reviewing data types and identifying columns that may need conversion
- Checking inconsistent formatting in categorical/text fields
- Detecting suspicious values in dates, coordinates, and key business fields
- Creating a structured data issues report for future cleaning and validation

## Analytical Approach

This notebook does not clean the dataset yet.

The goal is to detect, measure, explain, and document potential data issues so they can be handled properly in later DataGuard cleaning steps.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Imports complete.")

Imports complete.


In [2]:
project_root = Path.cwd()

# If this notebook is running from the notebooks folder, move one level up to DataGuard
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "nyc_311_raw_100k.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)

df.head()

Dataset loaded successfully.
Rows and columns: (100000, 44)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,69486831,2026-06-26T02:06:32.000,NaN,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Club/Bar/Restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,0 Unspecified,NaN,Unspecified,NaN,Unspecified,991542.0,189307.0,ONLINE,Unspecified,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.686278,-73.973707,POINT (-73.9737071634 40.686277751362)
1,69492768,2026-06-26T02:06:05.000,NaN,TLC,Taxi and Limousine Commission,For Hire Vehicle Complaint,Driver Complaint - Non Passenger,Unsafe Driving,Street,11211.0,180 UNION AVENUE,UNION AVENUE,BROADWAY,JOHNSON AVENUE,BROADWAY,JOHNSON AVENUE,ADDRESS,BROOKLYN,UNION AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,34.0,Precinct 90,3.030580e+09,BROOKLYN,998041.0,196408.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,"180 UNION AVENUE, BROOKLYN, NY, 11211",NaN,NaN,NaN,NaN,40.705761,-73.950259,POINT (-73.950259119641 40.705760603484)
2,69482343,2026-06-26T02:05:24.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10013.0,118 MULBERRY STREET,MULBERRY STREET,CANAL STREET,HESTER STREET,CANAL STREET,HESTER STREET,ADDRESS,NEW YORK,MULBERRY STREET,NaN,In Progress,NaN,NaN,NaN,02 MANHATTAN,1.0,Precinct 5,1.002050e+09,MANHATTAN,984753.0,200765.0,ONLINE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.717730,-73.998185,POINT (-73.998185472064 40.717730264942)
3,69488306,2026-06-26T02:03:49.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11249.0,96 WYTHE AVENUE,WYTHE AVENUE,NORTH 11 STREET,NORTH 10 STREET,NORTH 11 STREET,NORTH 10 STREET,ADDRESS,BROOKLYN,WYTHE AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,33.0,Precinct 94,3.022950e+09,BROOKLYN,995800.0,202137.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721489,-73.958332,POINT (-73.958332047132 40.721488549007)
4,69482692,2026-06-26T02:03:29.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,NaN,Street/Sidewalk,11421.0,85 STREET,85 STREET,85 STREET,88 ROAD,85 STREET,88 ROAD,INTERSECTION,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,09 QUEENS,32.0,Precinct 102,NaN,QUEENS,1023600.0,190652.0,ONLINE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689885,-73.858108,POINT (-73.858107579574 40.689884995274)


## Issue Category 1: Duplicate Records

Duplicate records can create inaccurate counts and misleading analysis.

For NYC 311, each row should represent one service request. The `unique_key` column is especially important because it should uniquely identify each request.

In this section, I will check:

- Full-row duplicates
- Duplicate `unique_key` values

In [3]:
# Count full-row duplicate records
full_duplicate_count = df.duplicated().sum()

print("Full-row duplicate count:", full_duplicate_count)

Full-row duplicate count: 0


In [4]:
# Count duplicate request IDs
duplicate_unique_key_count = df["unique_key"].duplicated().sum()

print("Duplicate unique_key count:", duplicate_unique_key_count)

Duplicate unique_key count: 0


### Observation: Duplicate Request IDs

There are 0 duplicate `unique_key` values in this sample.

This is a positive data quality finding because `unique_key` should uniquely identify each NYC 311 service request.

In [5]:
# Start a list to collect data quality issues
issues = []

# Add full-row duplicate check result
issues.append({
    "issue_category": "Duplicates",
    "issue_name": "Full-row duplicate records",
    "column_name": "All columns",
    "issue_count": full_duplicate_count,
    "severity": "Low" if full_duplicate_count == 0 else "High",
    "description": "Checks whether any complete rows are repeated exactly.",
    "recommended_action": "No action needed." if full_duplicate_count == 0 else "Investigate and remove duplicate rows after validation."
})

# Add duplicate unique_key check result
issues.append({
    "issue_category": "Duplicates",
    "issue_name": "Duplicate unique_key values",
    "column_name": "unique_key",
    "issue_count": duplicate_unique_key_count,
    "severity": "Low" if duplicate_unique_key_count == 0 else "Critical",
    "description": "Checks whether the request ID uniquely identifies each 311 service request.",
    "recommended_action": "No action needed." if duplicate_unique_key_count == 0 else "Investigate duplicate request IDs before cleaning."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'}]

## Issue Category 2: Data Type Issues

Wrong data types can make analysis unreliable.

For NYC 311, date columns should be converted to datetime before time-based analysis. Some numeric-looking fields, such as `unique_key` and `incident_zip`, are identifiers rather than true numeric measurements.

In [6]:
# Review data types
df.dtypes

unique_key                          int64
created_date                       object
closed_date                        object
agency                             object
agency_name                        object
complaint_type                     object
descriptor                         object
descriptor_2                       object
location_type                      object
incident_zip                      float64
incident_address                   object
street_name                        object
cross_street_1                     object
cross_street_2                     object
intersection_street_1              object
intersection_street_2              object
address_type                       object
city                               object
landmark                           object
facility_type                      object
status                             object
due_date                           object
resolution_description             object
resolution_action_updated_date    

### Observation: Data Type Review

The date-related columns, including `created_date`, `closed_date`, `due_date`, and `resolution_action_updated_date`, are currently stored as object/text columns. These should be converted to datetime before reliable time-based analysis.

The `unique_key` column is stored as `int64`, which is technically acceptable, but it should be treated as an identifier rather than a numeric measurement.

The `incident_zip` column is stored as `float64`, likely because of missing values. ZIP codes are location identifiers, so this column may need to be treated as text later during cleaning.

The `latitude` and `longitude` columns are stored as `float64`, which is appropriate for coordinate values.

In [7]:
# Convert date columns for data issue checks
date_columns = [
    "created_date",
    "closed_date",
    "due_date",
    "resolution_action_updated_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

df[date_columns].dtypes

created_date                      datetime64[ns]
closed_date                       datetime64[ns]
due_date                          datetime64[ns]
resolution_action_updated_date    datetime64[ns]
dtype: object

### Observation: Date Type Conversion

The date-related columns were successfully converted from object/text format to datetime format.

This allows reliable time-based data quality checks, such as checking date ranges and identifying records where `closed_date` occurs before `created_date`.

In [8]:
# Add data type issue findings to the issues list
issues.append({
    "issue_category": "Data Types",
    "issue_name": "Date columns loaded as object/text",
    "column_name": "created_date, closed_date, due_date, resolution_action_updated_date",
    "issue_count": len(date_columns),
    "severity": "Medium",
    "description": "Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.",
    "recommended_action": "Convert date columns to datetime during the cleaning step using pd.to_datetime(errors='coerce')."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

In [9]:
# Preview incident_zip values
df["incident_zip"].head(10)

0        NaN
1    11211.0
2    10013.0
3    11249.0
4    11421.0
5    11205.0
6    10456.0
7    11233.0
8    11205.0
9    11229.0
Name: incident_zip, dtype: float64

### Observation: ZIP Code Data Type

The `incident_zip` column is stored as `float64`, and ZIP code values appear with decimal formatting such as `11211.0`.

This is a data type and formatting concern because ZIP codes are location identifiers, not numeric measurements. This column may need to be converted to a cleaned string format during a later cleaning step.

In [10]:
# Add incident_zip data type issue to the issues list
issues.append({
    "issue_category": "Data Types",
    "issue_name": "incident_zip loaded as float",
    "column_name": "incident_zip",
    "issue_count": df["incident_zip"].notnull().sum(),
    "severity": "Medium",
    "description": "incident_zip is stored as float64, causing ZIP codes to appear with decimal formatting even though ZIP codes are identifiers.",
    "recommended_action": "Convert incident_zip to a cleaned string format during the cleaning step."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

## Issue Category 3: Inconsistent Formatting

Inconsistent formatting happens when values that should mean the same thing are written differently.

Examples include extra spaces, inconsistent capitalization, unexpected category labels, or mixed geographic levels.

This can affect analysis because pandas treats differently formatted values as separate categories.

In [11]:
# Check borough values
df["borough"].value_counts(dropna=False)

borough
BROOKLYN         31657
QUEENS           25993
BRONX            19431
MANHATTAN        18810
STATEN ISLAND     3983
Unspecified        126
Name: count, dtype: int64

### Observation: Borough Formatting

The `borough` column appears mostly standardized. The main borough values are written consistently in uppercase, such as `BROOKLYN`, `QUEENS`, `BRONX`, `MANHATTAN`, and `STATEN ISLAND`.

There are 126 records labeled as `Unspecified`. This is not a formatting error, but it may need review later because borough is an important location field for analysis.

In [12]:
# Add borough formatting check to issues list
unspecified_borough_count = (df["borough"] == "Unspecified").sum()

issues.append({
    "issue_category": "Formatting",
    "issue_name": "Unspecified borough values",
    "column_name": "borough",
    "issue_count": unspecified_borough_count,
    "severity": "Low" if unspecified_borough_count == 0 else "Medium",
    "description": "Checks for records where borough is labeled as Unspecified instead of a specific NYC borough.",
    "recommended_action": "Review Unspecified borough records during location data quality checks."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

In [13]:
# Check status values
df["status"].value_counts(dropna=False)

status
Closed         72749
Open           14206
In Progress    12183
Started          448
Assigned         348
Pending           66
Name: count, dtype: int64

### Observation: Status Formatting

The `status` column appears standardized. The values are consistently written as `Closed`, `Open`, `In Progress`, `Started`, `Assigned`, and `Pending`.

There are no obvious formatting issues such as inconsistent capitalization, extra spaces, or duplicate status labels.

In [14]:
# Add status formatting check to issues list
unexpected_status_count = 0

issues.append({
    "issue_category": "Formatting",
    "issue_name": "Unexpected status values",
    "column_name": "status",
    "issue_count": unexpected_status_count,
    "severity": "Low",
    "description": "Checks whether status values contain inconsistent or unexpected labels.",
    "recommended_action": "No action needed based on current status values."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

In [15]:
# Check top city values
df["city"].value_counts(dropna=False).head(20)

city
BROOKLYN            30117
BRONX               18539
NEW YORK            16510
NaN                  5999
STATEN ISLAND        3722
JAMAICA              2414
ASTORIA              2087
FLUSHING             1681
QUEENS               1505
FRESH MEADOWS        1160
RIDGEWOOD            1095
LONG ISLAND CITY      999
FAR ROCKAWAY          961
CORONA                922
MANHATTAN             836
JACKSON HEIGHTS       748
WOODSIDE              716
OZONE PARK            716
SOUTH OZONE PARK      714
EAST ELMHURST         630
Name: count, dtype: int64

### Observation: City Field Consistency

The `city` column contains a mix of borough names, city names, and neighborhood/location names.

Examples include borough-level values such as `BROOKLYN`, `BRONX`, `QUEENS`, `MANHATTAN`, and `STATEN ISLAND`, as well as neighborhood-level values such as `JAMAICA`, `ASTORIA`, `FLUSHING`, `RIDGEWOOD`, and `LONG ISLAND CITY`.

The column also contains 5,999 missing values.

This suggests that the `city` column may not represent one consistent geographic level. It should be reviewed before using it for location-based analysis.

In [16]:
# Add city consistency issue to issues list
city_missing_count = df["city"].isnull().sum()

issues.append({
    "issue_category": "Formatting / Consistency",
    "issue_name": "Mixed geographic levels in city field",
    "column_name": "city",
    "issue_count": city_missing_count,
    "severity": "Medium",
    "description": "The city column contains a mix of borough names, city names, neighborhood/location names, and missing values.",
    "recommended_action": "Review city values and decide whether to standardize, map to boroughs, or use borough/ZIP/location fields for geographic analysis."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

## Issue Category 4: Suspicious Values and Business Logic Checks

Suspicious values are values that may technically exist in the dataset but do not make sense based on business logic.

For NYC 311, one important business logic check is whether a request appears to be closed before it was created. This would create a negative resolution time and should be flagged for review.

In [17]:
# Calculate resolution time for records with created_date and closed_date
df["resolution_time"] = df["closed_date"] - df["created_date"]

negative_resolution_count = (df["resolution_time"] < pd.Timedelta(0)).sum()

print("Negative resolution time count:", negative_resolution_count)

Negative resolution time count: 39


### Observation: Negative Resolution Time

There are 39 records where `closed_date` occurs before `created_date`.

This creates a negative resolution time, which is suspicious because a service request should not be closed before it is created.

These records should be flagged for review before using resolution-time analysis.

In [18]:
# Add negative resolution time issue to issues list
issues.append({
    "issue_category": "Suspicious Values",
    "issue_name": "Negative resolution time",
    "column_name": "created_date, closed_date",
    "issue_count": negative_resolution_count,
    "severity": "High" if negative_resolution_count > 0 else "Low",
    "description": "Checks whether closed_date occurs before created_date, creating an invalid negative resolution time.",
    "recommended_action": "Flag these records for review before using resolution-time analysis." if negative_resolution_count > 0 else "No action needed."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

In [19]:
# Inspect records with negative resolution time
negative_resolution_records = df[df["resolution_time"] < pd.Timedelta(0)]

negative_resolution_records[
    [
        "unique_key",
        "created_date",
        "closed_date",
        "resolution_time",
        "agency",
        "complaint_type",
        "status",
        "borough"
    ]
].head(10)

,unique_key,created_date,closed_date,resolution_time,agency,complaint_type,status,borough
19483,69471158,2026-06-24 09:55:00,2026-06-23 09:54:00,-2 days +23:59:00,DOT,Street Light Condition,Pending,BRONX
19491,69475768,2026-06-24 09:54:00,2026-06-23 09:53:00,-2 days +23:59:00,DOT,Street Light Condition,Pending,BRONX
19560,69469646,2026-06-24 09:46:00,2026-06-22 09:45:00,-3 days +23:59:00,DOT,Street Light Condition,Pending,QUEENS
19858,69478717,2026-06-24 09:23:00,2026-06-23 09:22:00,-2 days +23:59:00,DOT,Street Light Condition,Pending,BRONX
30057,69460859,2026-06-23 09:57:00,2026-06-19 09:57:00,-4 days +00:00:00,DOT,Street Light Condition,Pending,BRONX
38081,69454957,2026-06-22 13:36:00,2026-06-12 13:36:00,-10 days +00:00:00,DOT,Street Light Condition,Pending,QUEENS
38093,69454958,2026-06-22 13:35:00,2026-06-11 13:35:00,-11 days +00:00:00,DOT,Street Light Condition,Pending,QUEENS
38105,69454959,2026-06-22 13:34:00,2026-06-11 13:34:00,-11 days +00:00:00,DOT,Street Light Condition,Pending,QUEENS
38106,69449375,2026-06-22 13:34:00,2026-06-11 13:33:00,-12 days +23:59:00,DOT,Street Light Condition,Pending,QUEENS
38141,69453548,2026-06-22 13:31:00,2026-06-11 13:31:00,-11 days +00:00:00,DOT,Street Light Condition,Pending,QUEENS


### Observation: Negative Resolution Record Patterns

The records with negative resolution time appear to be associated with the `DOT` agency and the `Street Light Condition` complaint type. The sample records are marked as `Pending`, even though they contain `closed_date` values earlier than `created_date`.

This suggests that the issue may be related to timestamp handling, workflow status logic, or how closure-related dates are recorded for this complaint type.

These records should be flagged for review before using `closed_date` to calculate resolution-time metrics.

## Coordinate Range Checks

Latitude and longitude values are important for location-based analysis.

For NYC 311 records, coordinates should generally fall within a reasonable NYC-area range. Values far outside this range may indicate invalid or unreliable location data.

In [20]:
# Review latitude and longitude summary statistics
df[["latitude", "longitude"]].describe()

,latitude,longitude
count,97965.000000,97965.000000
mean,40.731667,-73.921692
std,0.084657,0.078415
min,40.499093,-74.253920
25%,40.672243,-73.966883
50%,40.723333,-73.925180
75%,40.807521,-73.876304
max,40.912040,-73.700723


### Observation: Coordinate Ranges

The latitude and longitude values appear to fall within a reasonable NYC-area range.

The latitude values range from approximately 40.50 to 40.91, and the longitude values range from approximately -74.25 to -73.70. There are no obvious extreme coordinate values such as 0, 999, or values far outside the NYC region.

This suggests that the available coordinate values are generally reasonable for location-based analysis.

In [21]:
# Check for coordinates outside a broad NYC-area range
invalid_coordinate_count = df[
    (
        (df["latitude"].notnull()) &
        (df["longitude"].notnull()) &
        (
            (df["latitude"] < 40.0) |
            (df["latitude"] > 41.0) |
            (df["longitude"] < -75.0) |
            (df["longitude"] > -73.0)
        )
    )
].shape[0]

print("Invalid coordinate count:", invalid_coordinate_count)

Invalid coordinate count: 0


### Observation: Invalid Coordinate Check

There are 0 records with latitude/longitude values outside the broad NYC-area range used in this check.

This is a positive data quality finding. While some records have missing coordinates, the available coordinate values do not appear to contain obvious extreme location errors.

In [22]:
# Add coordinate range check to issues list
issues.append({
    "issue_category": "Suspicious Values",
    "issue_name": "Coordinates outside broad NYC-area range",
    "column_name": "latitude, longitude",
    "issue_count": invalid_coordinate_count,
    "severity": "Low" if invalid_coordinate_count == 0 else "High",
    "description": "Checks whether available latitude/longitude values fall outside a broad expected NYC-area range.",
    "recommended_action": "No action needed for coordinate range." if invalid_coordinate_count == 0 else "Flag invalid coordinates for location data review."
})

issues

[{'issue_category': 'Duplicates',
  'issue_name': 'Full-row duplicate records',
  'column_name': 'All columns',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether any complete rows are repeated exactly.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Duplicates',
  'issue_name': 'Duplicate unique_key values',
  'column_name': 'unique_key',
  'issue_count': np.int64(0),
  'severity': 'Low',
  'description': 'Checks whether the request ID uniquely identifies each 311 service request.',
  'recommended_action': 'No action needed.'},
 {'issue_category': 'Data Types',
  'issue_name': 'Date columns loaded as object/text',
  'column_name': 'created_date, closed_date, due_date, resolution_action_updated_date',
  'issue_count': 4,
  'severity': 'Medium',
  'description': 'Date-related columns were loaded as object/text and required conversion to datetime for reliable time-based analysis.',
  'recommended_action': "Convert date columns to date

In [23]:
# Create final data issues report
issues_report = pd.DataFrame(issues)

issues_report

,issue_category,issue_name,column_name,issue_count,severity,description,recommended_action
0,Duplicates,Full-row duplicate records,All columns,0,Low,Checks whether any complete rows are repeated ...,No action needed.
1,Duplicates,Duplicate unique_key values,unique_key,0,Low,Checks whether the request ID uniquely identif...,No action needed.
2,Data Types,Date columns loaded as object/text,"created_date, closed_date, due_date, resolutio...",4,Medium,Date-related columns were loaded as object/tex...,Convert date columns to datetime during the cl...
3,Data Types,incident_zip loaded as float,incident_zip,99189,Medium,"incident_zip is stored as float64, causing ZIP...",Convert incident_zip to a cleaned string forma...
4,Formatting,Unspecified borough values,borough,126,Medium,Checks for records where borough is labeled as...,Review Unspecified borough records during loca...
5,Formatting,Unexpected status values,status,0,Low,Checks whether status values contain inconsist...,No action needed based on current status values.
6,Formatting / Consistency,Mixed geographic levels in city field,city,5999,Medium,The city column contains a mix of borough name...,Review city values and decide whether to stand...
7,Suspicious Values,Negative resolution time,"created_date, closed_date",39,High,Checks whether closed_date occurs before creat...,Flag these records for review before using res...
8,Suspicious Values,Coordinates outside broad NYC-area range,"latitude, longitude",0,Low,Checks whether available latitude/longitude va...,No action needed for coordinate range.


In [24]:
issues_report.shape

(9, 7)

In [25]:
# Count issues by category
issues_report["issue_category"].value_counts()

issue_category
Duplicates                  2
Data Types                  2
Formatting                  2
Suspicious Values           2
Formatting / Consistency    1
Name: count, dtype: int64

In [26]:
# Count issues by severity
issues_report["severity"].value_counts()

severity
Low       4
Medium    4
High      1
Name: count, dtype: int64

In [27]:
# Count issues by category
issues_report["issue_category"].value_counts()

issue_category
Duplicates                  2
Data Types                  2
Formatting                  2
Suspicious Values           2
Formatting / Consistency    1
Name: count, dtype: int64

### Data Issues Report Summary

The Day 4 data issues report contains 9 checks across duplicate records, data types, formatting/consistency, and suspicious values.

The report includes 4 low-severity checks, 4 medium-severity issues, and 1 high-severity issue. The high-severity issue is related to negative resolution time, where some records have `closed_date` earlier than `created_date`.

Overall, the dataset has no duplicate rows or duplicate `unique_key` values, which is a positive finding. However, there are issues that should be reviewed later, including date columns originally loaded as text, ZIP codes loaded as floats, mixed geographic levels in the `city` field, and negative resolution-time records.

In [28]:
# Export Day 4 data issues report
reports_folder = project_root / "reports"
reports_folder.mkdir(exist_ok=True)

issues_report_path = reports_folder / "data_issues_report_day4.csv"

issues_report.to_csv(issues_report_path, index=False)

print("Data issues report saved to:", issues_report_path)

Data issues report saved to: c:\Users\samee\DataGuard\reports\data_issues_report_day4.csv


In [29]:
# Verify exported Day 4 report exists
issues_report_path.exists()

True

# Day 4 Summary

In this notebook, I created a data issues report for the NYC 311 dataset.

The goal was to detect data quality issues beyond missing values, including duplicate records, data type problems, inconsistent formatting, and suspicious values.

## Work Completed

- Loaded the 100,000-row NYC 311 dataset sample
- Checked for full-row duplicate records
- Checked for duplicate `unique_key` values
- Reviewed column data types
- Converted date columns to datetime format for quality checks
- Identified date columns that were originally loaded as object/text
- Identified `incident_zip` as a formatting/data type concern because ZIP codes were loaded as floats
- Checked borough formatting
- Checked status formatting
- Reviewed the `city` column for consistency issues
- Created `resolution_time` using `closed_date - created_date`
- Identified records with negative resolution time
- Checked latitude and longitude ranges for suspicious coordinate values
- Created a structured Day 4 data issues report
- Exported the final report as `reports/data_issues_report_day4.csv`

## Key Findings

- There are 0 full-row duplicate records.
- There are 0 duplicate `unique_key` values.
- Date-related columns were originally loaded as object/text and need datetime conversion for reliable time-based analysis.
- `incident_zip` is stored as `float64`, which is a concern because ZIP codes are identifiers, not numeric measurements.
- The `borough` and `status` columns appear mostly standardized.
- The `city` column contains mixed geographic levels, including borough names and neighborhood/location names.
- There are 39 records with negative resolution time, meaning `closed_date` occurs before `created_date`.
- Available latitude and longitude values fall within a broad NYC-area range.

## Output Created

The Day 4 data issues report was saved to:

`reports/data_issues_report_day4.csv`

## Next Step

The next step is Day 5: writing the first auto-fix functions to standardize values, parse dates, and remove duplicates automatically.